<a href="https://colab.research.google.com/github/srangisetti27/TS/blob/main/tune_and_ensemble_breast_cancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Optimization: Tune & Ensemble a Classifier

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data): 569 tumour samples, 30 numeric features computed from cell-nucleus images. Target: `diagnosis` (M = malignant, B = benign).

**Plan**
- Load and check the data
- Train/test split (the test set stays untouched until the end)
- **Baseline**: Decision Tree, evaluated with **5-fold Cross Validation**
- Tune it with **GridSearchCV**
- Tune it with **RandomizedSearchCV**
- Build an **Ensemble**: Random Forest
- Compare everything in **one table**

**Why a Decision Tree?** A single tree overfits easily (it can memorise the training set), so it gains a lot from tuning. The Random Forest ensemble is built from many trees, so it shows how an ensemble improves on the same base model.

## 1. Imports

In [1]:
import os
import time
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     cross_validate, GridSearchCV, RandomizedSearchCV)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
RANDOM_STATE = 42   # fixed seed so results are reproducible

## 2. Load the data

The notebook looks for Kaggle's `data.csv` next to the notebook, or in the Kaggle input folder if you run it on Kaggle.
If neither is found, it falls back to scikit-learn's built-in copy of the **same 569 UCI records**
and rebuilds the Kaggle layout (same column names, `diagnosis` as M/B), so every cell below runs the same either way.

In [2]:
LOCAL_PATH = "data.csv"
KAGGLE_PATH = "/kaggle/input/breast-cancer-wisconsin-data/data.csv"

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print("Loaded Kaggle data.csv")
elif os.path.exists(KAGGLE_PATH):
    df = pd.read_csv(KAGGLE_PATH)
    print("Loaded Kaggle data.csv from /kaggle/input")
else:
    bc = load_breast_cancer(as_frame=True)
    df = bc.frame.copy()
    df["diagnosis"] = np.where(df["target"] == 0, "M", "B")   # in sklearn, 0 = malignant
    df = df.drop(columns="target")
    print("data.csv not found: using scikit-learn's copy of the same dataset")

# Kaggle's file has an 'id' column and an empty 'Unnamed: 32' column; neither is a feature
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")

# Give both sources the same column names, e.g. 'radius_mean', 'concave_points_worst'
df.columns = (df.columns.str.replace(r"^mean (.*)$", r"\1_mean", regex=True)
                        .str.replace(r"^(.*) error$", r"\1_se", regex=True)
                        .str.replace(r"^worst (.*)$", r"\1_worst", regex=True)
                        .str.replace(" ", "_"))

print("Shape:", df.shape)
df.head()

data.csv not found: using scikit-learn's copy of the same dataset
Shape: (569, 31)


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave_points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,M
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,M
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,M
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,M
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,M


### Quick data checks

In [3]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print()
print("Class balance:")
print(df["diagnosis"].value_counts())
print()
print(df["diagnosis"].value_counts(normalize=True).round(3))

Missing values: 0
Duplicate rows: 0

Class balance:
diagnosis
B    357
M    212
Name: count, dtype: int64

diagnosis
B    0.627
M    0.373
Name: proportion, dtype: float64


No missing values and no duplicates. The classes are somewhat imbalanced (about 63% benign, 37% malignant),
so we use **stratified** splits to keep that ratio in every fold, and we look at more than just accuracy.

## 3. Features, target and train/test split

- Target: malignant = **1** (positive class), benign = **0**. Missing a malignant tumour (a false negative) is the costly mistake, so **recall** and **F1** for class 1 matter most.
- **No scaling needed.** Decision Trees and Random Forests split on thresholds, so feature scale does not affect them.
- **Avoiding data leakage:** we split first. All cross validation and tuning happens on the **training set only**; the test set is used once at the end.

In [4]:
X = df.drop(columns="diagnosis")
y = df["diagnosis"].map({"M": 1, "B": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Malignant share - train: %.3f  test: %.3f" % (y_train.mean(), y_test.mean()))

Train: (455, 30)  Test: (114, 30)
Malignant share - train: 0.374  test: 0.368


## 4. Evaluation setup

Every model is scored the same way:
- **5-fold Stratified Cross Validation** on the training set, using the **same 5 folds** for every model (fixed `random_state`), so the comparison is fair
- one final score on the **held-out test set**

The helper function below does both and returns one row for the comparison table.

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = ["accuracy", "precision", "recall", "f1", "roc_auc"]

def evaluate(name, model, search_fits=0, tuning_time=0.0):
    """5-fold CV on the training set, then fit on the full training set and score on the test set."""
    cv_res = cross_validate(model, X_train, y_train, cv=cv, scoring=SCORING)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    return {
        "Model": name,
        "CV Accuracy (mean)": cv_res["test_accuracy"].mean(),
        "CV Accuracy (std)": cv_res["test_accuracy"].std(),
        "CV Recall": cv_res["test_recall"].mean(),
        "CV F1": cv_res["test_f1"].mean(),
        "CV ROC-AUC": cv_res["test_roc_auc"].mean(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred),
        "Test Recall": recall_score(y_test, y_pred),
        "Test F1": f1_score(y_test, y_pred),
        "Test ROC-AUC": roc_auc_score(y_test, y_prob),
        "Missed Malignant (FN)": fn,
        "False Alarms (FP)": fp,
        "Search Fits": search_fits,
        "Tuning Time (s)": tuning_time,
    }

## 5. Baseline: Decision Tree with 5-fold Cross Validation

Default settings: the tree keeps splitting until every leaf is pure.

In [6]:
baseline = DecisionTreeClassifier(random_state=RANDOM_STATE)

fold_scores = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="accuracy")
print("Accuracy per fold:", fold_scores.round(4))
print("Mean: %.4f   Std: %.4f" % (fold_scores.mean(), fold_scores.std()))

Accuracy per fold: [0.8901 0.967  0.8901 0.9121 0.956 ]
Mean: 0.9231   Std: 0.0326


**Overfitting check:** compare training accuracy with validation accuracy across the same 5 folds.
A large gap means the tree is memorising the training data.

In [7]:
over = cross_validate(baseline, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
print("Mean train accuracy:      %.4f" % over["train_score"].mean())
print("Mean validation accuracy: %.4f" % over["test_score"].mean())

baseline_fit = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
print("Fully grown tree -> depth:", baseline_fit.get_depth(), " leaves:", baseline_fit.get_n_leaves())

Mean train accuracy:      1.0000
Mean validation accuracy: 0.9231
Fully grown tree -> depth: 8  leaves: 24


In [8]:
baseline_res = evaluate("Decision Tree (baseline)", baseline)
pd.Series(baseline_res).to_frame("Baseline")

,Baseline
Model,Decision Tree (baseline)
CV Accuracy (mean),0.923077
CV Accuracy (std),0.032599
CV Recall,0.894118
CV F1,0.896184
CV ROC-AUC,0.917234
Test Accuracy,0.929825
Test Precision,0.904762
Test Recall,0.904762
Test F1,0.904762


## 6. Tuning with GridSearchCV

GridSearchCV tries **every combination** in the grid, using the same 5 folds.
We optimise **F1 for the malignant class**, which balances catching cancers (recall) against false alarms (precision).

The hyperparameters all limit how far the tree can grow, which is the main cure for overfitting:
- `max_depth`: maximum number of levels
- `min_samples_split`: a node needs at least this many samples to be split
- `min_samples_leaf`: every leaf must keep at least this many samples
- `criterion`: how split quality is measured (Gini impurity or entropy)

In [9]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 3, 4, 5, 6, 8],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
}
n_combos = 2 * 6 * 4 * 4
print("Combinations:", n_combos, "-> fits:", n_combos * 5)

grid = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),
                    param_grid, cv=cv, scoring="f1", n_jobs=-1)

start = time.time()
grid.fit(X_train, y_train)
grid_time = time.time() - start

print("Best params:", grid.best_params_)
print("Best CV F1: %.4f" % grid.best_score_)
print("Search time: %.1f s" % grid_time)

Combinations: 192 -> fits: 960
Best params: {'criterion': 'entropy', 'max_depth': 4, 'min_samples_leaf': 2, 'min_samples_split': 2}
Best CV F1: 0.9163
Search time: 15.7 s


In [10]:
grid_res = evaluate("Decision Tree + GridSearchCV", grid.best_estimator_,
                    search_fits=n_combos * 5, tuning_time=grid_time)

## 7. Tuning with RandomizedSearchCV

RandomizedSearchCV samples a **fixed number of random combinations** (`n_iter`) from ranges or distributions.
It can explore a **wider** space at **lower** cost than a grid. Here it also searches two settings the grid did not:
- `max_features`: how many features each split may consider
- `ccp_alpha`: cost-complexity pruning, which cuts back branches that add little

In [11]:
param_dist = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None] + list(range(2, 16)),
    "min_samples_split": randint(2, 41),     # integers 2-40
    "min_samples_leaf": randint(1, 21),      # integers 1-20
    "max_features": [None, "sqrt", "log2"],
    "ccp_alpha": uniform(0.0, 0.03),         # floats 0.0-0.03
}
N_ITER = 60
print("Sampled combinations:", N_ITER, "-> fits:", N_ITER * 5)

rand = RandomizedSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),
                          param_dist, n_iter=N_ITER, cv=cv, scoring="f1",
                          random_state=RANDOM_STATE, n_jobs=-1)

start = time.time()
rand.fit(X_train, y_train)
rand_time = time.time() - start

print("Best params:", rand.best_params_)
print("Best CV F1: %.4f" % rand.best_score_)
print("Search time: %.1f s" % rand_time)

Sampled combinations: 60 -> fits: 300
Best params: {'ccp_alpha': np.float64(0.009241823755571676), 'criterion': 'entropy', 'max_depth': 12, 'max_features': 'log2', 'min_samples_leaf': 19, 'min_samples_split': 23}
Best CV F1: 0.9119
Search time: 4.2 s


In [12]:
rand_res = evaluate("Decision Tree + RandomizedSearchCV", rand.best_estimator_,
                    search_fits=N_ITER * 5, tuning_time=rand_time)

## 8. Ensemble: Random Forest

A Random Forest trains many decision trees, each on a **bootstrap sample** of the rows (bagging),
and each split only considers a **random subset of features**. The trees make different mistakes,
so averaging their votes cancels much of the variance that makes a single tree overfit.

In [13]:
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf_res = evaluate("Random Forest (ensemble, 300 trees)", rf)

## 9. Comparison table

CV columns = mean over the same 5 training folds; Test columns = the held-out 20% (114 samples, 42 malignant).

In [14]:
results = pd.DataFrame([baseline_res, grid_res, rand_res, rf_res]).set_index("Model")
results.round(4)

,CV Accuracy (mean),CV Accuracy (std),CV Recall,CV F1,CV ROC-AUC,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC,Missed Malignant (FN),False Alarms (FP),Search Fits,Tuning Time (s)
Model,,,,,,,,,,,,,,
Decision Tree (baseline),0.9231,0.0326,0.8941,0.8962,0.9172,0.9298,0.9048,0.9048,0.9048,0.9246,4,4,0,0.0000
Decision Tree + GridSearchCV,0.9407,0.0215,0.8824,0.9163,0.9472,0.9298,1.0000,0.8095,0.8947,0.9563,8,0,960,15.6534
Decision Tree + RandomizedSearchCV,0.9341,0.0120,0.9118,0.9119,0.9678,0.9561,0.9744,0.9048,0.9383,0.9722,4,1,300,4.1578
"Random Forest (ensemble, 300 trees)",0.9604,0.0164,0.9353,0.9461,0.9880,0.9737,1.0000,0.9286,0.9630,0.9944,3,0,0,0.0000


**Best hyperparameters found**

In [15]:
pd.DataFrame({
    "GridSearchCV": pd.Series(grid.best_params_),
    "RandomizedSearchCV": pd.Series(rand.best_params_),
}).fillna("not searched (default)").astype(str)

,GridSearchCV,RandomizedSearchCV
ccp_alpha,not searched (default),0.009241823755571676
criterion,entropy,entropy
max_depth,4,12
max_features,not searched (default),log2
min_samples_leaf,2,19
min_samples_split,2,23


**Confusion matrices on the test set.** Rows = actual (benign, malignant), columns = predicted (benign, malignant).
The bottom-left cell is malignant tumours predicted as benign.

In [16]:
print("Baseline Decision Tree\n", confusion_matrix(y_test, baseline.predict(X_test)), "\n")
print("GridSearchCV Decision Tree\n", confusion_matrix(y_test, grid.best_estimator_.predict(X_test)), "\n")
print("RandomizedSearchCV Decision Tree\n", confusion_matrix(y_test, rand.best_estimator_.predict(X_test)), "\n")
print("Random Forest\n", confusion_matrix(y_test, rf.predict(X_test)))

Baseline Decision Tree
 [[68  4]
 [ 4 38]] 

GridSearchCV Decision Tree
 [[72  0]
 [ 8 34]] 

RandomizedSearchCV Decision Tree
 [[71  1]
 [ 4 38]] 

Random Forest
 [[72  0]
 [ 3 39]]


**Top 10 features used by the Random Forest**

In [17]:
pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10).round(4)

,0
perimeter_worst,0.1431
area_worst,0.1424
concave_points_worst,0.1130
concave_points_mean,0.0887
radius_worst,0.0826
radius_mean,0.0610
perimeter_mean,0.0529
area_mean,0.0438
concavity_mean,0.0398
area_se,0.0322


## 10. Conclusions

**1. The baseline overfits.** The fully grown tree (depth 8, 24 leaves) scores 1.000 on the training folds but 0.923 on the validation folds, and its fold scores swing from 0.890 to 0.967 (std 0.033).

**2. Both searches improved the tree under cross validation.** Limiting how far the tree can grow raised CV accuracy from 0.923 to 0.941 (Grid) and 0.934 (Randomized), and CV ROC-AUC from 0.917 to 0.947 and 0.968. Results also became steadier across folds (std 0.033 → 0.022 and 0.012).

**3. RandomizedSearchCV matched the grid for a third of the work.** It ran 300 fits against GridSearchCV's 960 (see the Tuning Time column) and still searched a wider space, including pruning (`ccp_alpha`) and `max_features`. Its best CV F1 was 0.912 against the grid's 0.916, a negligible gap.

**4. A single test set is noisy, so trust CV more.** The grid-tuned tree had perfect test precision but missed 8 of the 42 malignant tumours (recall 0.81), while the randomized tree missed 4. Part of this is real: the grid's shallow depth-4 tree trades recall for precision, and its CV recall (0.882) is the lowest of the four models. But with only 42 malignant test cases, each miss moves recall by about 2.4 points, so the small test set exaggerates the gap.
CV scores for the tuned models are also slightly optimistic, because their hyperparameters were chosen on those same folds. Nested cross validation would remove that bias.

**5. The Random Forest ensemble is the best model, with no tuning at all.** It has the highest CV and test accuracy, recall, F1 and ROC-AUC of the four models: CV accuracy 0.960, CV F1 0.946 and CV ROC-AUC 0.988. On the test set it reaches 0.974 accuracy, 0.963 F1 and 0.994 ROC-AUC, with 3 missed malignant cases and 0 false alarms. Averaging 300 trees that each see different rows and features cancels out the variance that makes a single tree unstable. Tuning one tree can only partly fix that variance.

**Model to pick: Random Forest.** Its most important features (worst perimeter, worst area, worst and mean concave points) all describe tumour size and how irregular its outline is.

**Possible next step:** even the forest misses 3 malignant tumours. A missed cancer costs far more than a false alarm. Lowering the decision threshold below 0.5, or tuning with `scoring="recall"`, would accept a few more false alarms in exchange for fewer missed cancers.